# GNN Oversquashing Experiment: Baseline Pipeline

In [43]:
!pip install -q torch_geometric

### Mount Files

In [44]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Imports & Setup

In [45]:
from collections import deque

import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from torch_geometric.utils import to_undirected


GRAPH_PATH = "/content/drive/MyDrive/Thesis /reddit_base_graph.pt"

TARGET_IDX = 0
HIDDEN_DIM = 64
NUM_LAYERS = 3
EPOCHS = 500
PATIENCE = 80
LR = 5e-3
WEIGHT_DECAY = 5e-4
SEED = 0
N_SEEDS = 100

torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

Device: cuda


## Load Graph

In [46]:
def load_graph_with_source_mask(path: str, target_idx: int):
    """
    Load the base graph, peel target off as y, and flag which nodes are
    actually source subreddits (have at least one outgoing edge).

    Non-source nodes have all-zero features from your preprocessing, so they
    must NOT be supervised on — but we keep them in the graph because they
    matter for message-passing structure.
    """
    graph = torch.load(path, weights_only=False)
    full_x = graph.x  # [N, 86]
    edge_index = graph.edge_index

    y = full_x[:, target_idx].clone()
    keep = [i for i in range(full_x.size(1)) if i != target_idx]
    x = full_x[:, keep].contiguous()

    # A "real source" appears at least once as edge_index[0].
    is_source = torch.zeros(full_x.size(0), dtype=torch.bool)
    is_source[edge_index[0].unique()] = True

    data = Data(x=x, y=y, edge_index=edge_index)
    data.num_nodes = full_x.size(0)
    data.is_source = is_source
    return data


data = load_graph_with_source_mask(GRAPH_PATH, target_idx=TARGET_IDX)
n_source = int(data.is_source.sum())
print(f"N={data.num_nodes}  x={tuple(data.x.shape)}  edges={data.edge_index.size(1)}")
print(f"Real source nodes: {n_source} ({100*n_source/data.num_nodes:.1f}%)")

N=35776  x=(35776, 85)  edges=286561
Real source nodes: 27863 (77.9%)


### Diagnostic of Target

In [47]:
def describe_target(y, mask, label="y"):
    y_sub = y[mask]
    print(f"\n=== {label} distribution (n={int(mask.sum())}) ===")
    print(f"  min={y_sub.min().item():.4f}  max={y_sub.max().item():.4f}")
    print(f"  mean={y_sub.mean().item():.4f}  std={y_sub.std().item():.4f}")
    print(f"  median={y_sub.median().item():.4f}")
    print(f"  exactly zero: {(y_sub == 0).sum().item()} / {len(y_sub)}")
    print("  deciles:")
    qs = torch.quantile(y_sub, torch.linspace(0, 1, 11))
    for i, q in enumerate(qs.tolist()):
        print(f"    {i*10:3d}%: {q:.4f}")


describe_target(data.y, data.is_source, label=f"feature {TARGET_IDX}")



=== feature 0 distribution (n=27863) ===
  min=49.0000  max=40276.0000
  mean=1711.0648  std=2773.7258
  median=912.2500
  exactly zero: 0 / 27863
  deciles:
      0%: 49.0000
     10%: 196.8857
     20%: 344.0000
     30%: 502.0000
     40%: 696.0000
     50%: 912.2500
     60%: 1194.4669
     70%: 1581.5668
     80%: 2240.7644
     90%: 3746.5464
    100%: 40276.0000


# BFS

In [48]:
def pick_source_node(edge_index: torch.Tensor, num_nodes: int) -> int:
    out_deg = torch.zeros(num_nodes, dtype=torch.long)
    out_deg.scatter_add_(0, edge_index[0], torch.ones(edge_index.size(1), dtype=torch.long))
    return int(out_deg.argmax().item())


def bfs_hop_distances(edge_index: torch.Tensor, num_nodes: int, source: int) -> torch.Tensor:
    """BFS over the undirected view. Unreachable nodes get -1."""
    undirected = to_undirected(edge_index, num_nodes=num_nodes)
    row, col = undirected
    order = row.argsort()
    row_sorted, col_sorted = row[order], col[order]
    ptr = torch.zeros(num_nodes + 1, dtype=torch.long)
    ptr[1:] = torch.bincount(row_sorted, minlength=num_nodes).cumsum(0)

    dist = torch.full((num_nodes,), -1, dtype=torch.long)
    dist[source] = 0
    q = deque([source])
    while q:
        u = q.popleft()
        d = dist[u].item()
        for v in col_sorted[ptr[u]:ptr[u + 1]].tolist():
            if dist[v] == -1:
                dist[v] = d + 1
                q.append(v)
    return dist


anchor = pick_source_node(data.edge_index, data.num_nodes)
print(f"Source anchor: node {anchor} (highest out-degree)")

dist = bfs_hop_distances(data.edge_index, data.num_nodes, anchor)

print("\nDistance histogram (all nodes):")
for d in range(0, 8):
    n = int((dist == d).sum())
    if n > 0:
        print(f"  dist=={d}: {n}")
print(f"  dist>=8:   {int((dist >= 8).sum())}")
print(f"  unreached: {int((dist == -1).sum())}")

print("\nDistance histogram (real source nodes only — what we'll supervise on):")
for d in range(0, 8):
    n = int(((dist == d) & data.is_source).sum())
    if n > 0:
        print(f"  dist=={d}: {n}")
print(f"  dist>=8:   {int(((dist >= 8) & data.is_source).sum())}")



Source anchor: node 122 (highest out-degree)

Distance histogram (all nodes):
  dist==0: 1
  dist==1: 1598
  dist==2: 21225
  dist==3: 10794
  dist==4: 964
  dist==5: 86
  dist==6: 3
  dist>=8:   0
  unreached: 1105

Distance histogram (real source nodes only — what we'll supervise on):
  dist==0: 1
  dist==1: 1454
  dist==2: 17876
  dist==3: 7269
  dist==4: 613
  dist==5: 54
  dist==6: 3
  dist>=8:   0


### Masks

In [49]:
def build_distance_masks(dist: torch.Tensor, is_source: torch.Tensor):
    """
    train: 0–1 hop AND real source  (core community)
    val:   2 hop   AND real source  (narrow bridges)
    test:  3+ hop  AND real source  (distant — where oversquashing should bite)

    Non-source nodes (zero-feature placeholders) are excluded from all splits
    but remain in the graph for message passing.
    """
    train_mask = (dist >= 0) & (dist <= 1) & is_source
    val_mask = (dist == 2) & is_source
    test_mask = (dist >= 3) & is_source
    return train_mask, val_mask, test_mask


train_mask, val_mask, test_mask = build_distance_masks(dist, data.is_source)
print(f"\nSplit sizes (real sources only):")
print(f"  train (0-1 hop): {int(train_mask.sum())}")
print(f"  val   (2 hop):   {int(val_mask.sum())}")
print(f"  test  (3+ hop):  {int(test_mask.sum())}")



Split sizes (real sources only):
  train (0-1 hop): 1455
  val   (2 hop):   17876
  test  (3+ hop):  7939


### Graph Sage Baseline

In [50]:
class GraphSAGEBaseline(torch.nn.Module):
    """3-layer SAGE → 3-hop receptive field, matching the test boundary."""

    def __init__(self, in_dim: int, hidden_dim: int = 128, num_layers: int = 3, dropout: float = 0.3):
        super().__init__()
        self.dropout = dropout
        self.convs = torch.nn.ModuleList()
        self.convs.append(SAGEConv(in_dim, hidden_dim))
        for _ in range(num_layers - 2):
            self.convs.append(SAGEConv(hidden_dim, hidden_dim))
        self.convs.append(SAGEConv(hidden_dim, hidden_dim))
        self.head = torch.nn.Linear(hidden_dim, 1)

    def forward(self, x, edge_index):
        for conv in self.convs[:-1]:
            x = conv(x, edge_index)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.convs[-1](x, edge_index)
        return self.head(x).squeeze(-1)


### Training & Eval

In [51]:
def standardize_with_train_stats(x, y, train_mask):
    """Normalize features and target using only train-node statistics."""
    x_mean = x[train_mask].mean(dim=0, keepdim=True)
    x_std = x[train_mask].std(dim=0, keepdim=True).clamp_min(1e-6)
    x_n = (x - x_mean) / x_std

    y_mean = y[train_mask].mean()
    y_std = y[train_mask].std().clamp_min(1e-6)
    y_n = (y - y_mean) / y_std
    return x_n, y_n, y_mean.item(), y_std.item()


@torch.no_grad()
def per_band_mse(model, data, dist, y_std):
    """MSE per distance band, scaled back to original units."""
    model.eval()
    pred = model(data.x, data.edge_index)
    results = []
    for d in [0, 1, 2, 3, 4]:
        band = (dist == d) & data.is_source
        if band.sum() == 0:
            continue
        mse = F.mse_loss(pred[band], data.y[band]).item() * (y_std ** 2)
        results.append((f"dist == {d}", int(band.sum()), mse))
    band = (dist >= 5) & data.is_source
    if band.sum() > 0:
        mse = F.mse_loss(pred[band], data.y[band]).item() * (y_std ** 2)
        results.append(("dist >= 5", int(band.sum()), mse))
    return results


def constant_baseline_mse(y_raw, train_mask, eval_mask):
    """Predict the train-set mean for every node. This is the floor —
    any real model needs to beat this on the test set to be doing anything."""
    pred = y_raw[train_mask].mean()
    return ((y_raw[eval_mask] - pred) ** 2).mean().item()


def train_baseline(data, train_mask, val_mask, test_mask, dist, device=DEVICE):
    data = data.to(device)
    train_mask = train_mask.to(device)
    val_mask = val_mask.to(device)
    test_mask = test_mask.to(device)
    dist = dist.to(device)

    # Symmetrize edges for message passing (matches the BFS view of the graph).
    data.edge_index = to_undirected(data.edge_index, num_nodes=data.num_nodes)

    # Constant-baseline reference points (in raw units), computed BEFORE
    # standardization so we can compare apples to apples later.
    y_raw = data.y.clone()
    const_train = constant_baseline_mse(y_raw, train_mask, train_mask)
    const_val = constant_baseline_mse(y_raw, train_mask, val_mask)
    const_test = constant_baseline_mse(y_raw, train_mask, test_mask)
    print(f"\nConstant-predictor MSE (raw units):  "
          f"train={const_train:.4f}  val={const_val:.4f}  test={const_test:.4f}")
    print("Any GNN that doesn't beat these numbers is doing nothing useful.\n")

    # Standardize using train-only stats.
    data.x, data.y, _, y_std = standardize_with_train_stats(data.x, data.y, train_mask)

    model = GraphSAGEBaseline(
        in_dim=data.x.size(1), hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS
    ).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    masks = {"train": train_mask, "val": val_mask, "test": test_mask}

    @torch.no_grad()
    def eval_all():
        model.eval()
        pred = model(data.x, data.edge_index)
        out = {}
        for name, mask in masks.items():
            out[name] = F.mse_loss(pred[mask], data.y[mask]).item() * (y_std ** 2)
        return out

    best_val = float("inf")
    best_state = None
    epochs_since_improve = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        opt.zero_grad()
        pred = model(data.x, data.edge_index)
        loss = F.mse_loss(pred[train_mask], data.y[train_mask])
        loss.backward()
        opt.step()

        if epoch % 10 == 0 or epoch == 1:
            scores = eval_all()
            print(
                f"epoch {epoch:3d} | "
                f"train MSE {scores['train']:.4f} | "
                f"val MSE {scores['val']:.4f} | "
                f"test MSE {scores['test']:.4f}"
            )
            if scores["val"] < best_val - 1e-5:
                best_val = scores["val"]
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                epochs_since_improve = 0
            else:
                epochs_since_improve += 10
                if epochs_since_improve >= PATIENCE:
                    print(f"early stop at epoch {epoch}")
                    break

    if best_state is not None:
        model.load_state_dict(best_state)

    print("\n=== Per-distance MSE (raw units, best-val checkpoint) ===")
    print(f"  {'band':<11s}  {'n':>6s}  {'MSE':>10s}  {'const baseline':>16s}")
    rows = per_band_mse(model, data, dist, y_std)
    for band_name, n, mse in rows:
        # constant baseline for this band
        if band_name.startswith("dist == "):
            d = int(band_name.split()[-1])
            band = (dist == d) & data.is_source
        else:
            band = (dist >= 5) & data.is_source
        cb = constant_baseline_mse(y_raw, train_mask, band)
        print(f"  {band_name:<11s}  {n:>6d}  {mse:>10.4f}  {cb:>16.4f}")

    return model, y_std


### Running & Training

In [52]:

import copy
import json

SEEDS = list(range(N_SEEDS))


data_original = Data(
    x=data.x.clone(),
    y=data.y.clone(),
    edge_index=data.edge_index.clone(),
)
data_original.num_nodes = data.num_nodes
data_original.is_source = data.is_source.clone()

all_runs = []

for seed in SEEDS:
    print(f"\n{'='*60}")
    print(f"  SEED {seed}")
    print(f"{'='*60}")
    torch.manual_seed(seed)


    data_run = Data(
        x=data_original.x.clone(),
        y=data_original.y.clone(),
        edge_index=data_original.edge_index.clone(),
    )
    data_run.num_nodes = data_original.num_nodes
    data_run.is_source = data_original.is_source.clone()


    model, y_std = train_baseline(data_run, train_mask, val_mask, test_mask, dist)

    rows = per_band_mse(model, data_run.to(DEVICE), dist.to(DEVICE), y_std)
    all_runs.append({"seed": seed, "rows": rows, "y_std": y_std})


print(f"\n{'='*60}")
print(f"  AGGREGATE ACROSS {len(SEEDS)} SEEDS")
print(f"{'='*60}\n")


import collections
band_mses = collections.defaultdict(list)
band_ns = {}
for run in all_runs:
    for band_name, n, mse in run["rows"]:
        band_mses[band_name].append(mse)
        band_ns[band_name] = n

print(f"  {'band':<11s}  {'n':>6s}  {'mean MSE':>12s}  {'std':>10s}  {'min':>10s}  {'max':>10s}")
summary_rows = []
for band_name in ["dist == 0", "dist == 1", "dist == 2", "dist == 3", "dist == 4", "dist >= 5"]:
    if band_name not in band_mses:
        continue
    mses = band_mses[band_name]
    n = band_ns[band_name]
    mean = sum(mses) / len(mses)
    std = (sum((m - mean) ** 2 for m in mses) / len(mses)) ** 0.5
    print(f"  {band_name:<11s}  {n:>6d}  {mean:>12.2f}  {std:>10.2f}  {min(mses):>10.2f}  {max(mses):>10.2f}")
    summary_rows.append({
        "band": band_name, "n": n,
        "mean_mse": mean, "std_mse": std,
        "min_mse": min(mses), "max_mse": max(mses),
        "all_mses": mses,
    })


results = {
    "model": "GraphSAGE_3layer",
    "target_idx": TARGET_IDX,
    "anchor": int(anchor),
    "seeds": SEEDS,
    "split_sizes": {
        "train": int(train_mask.sum()),
        "val": int(val_mask.sum()),
        "test": int(test_mask.sum()),
    },
    "per_band": summary_rows,
}
out_path = "/content/drive/MyDrive/Thesis /baseline_results.json"
with open(out_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"\nSaved to {out_path}")

Streaming output truncated to the last 5000 lines.
epoch 150 | train MSE 22719.1079 | val MSE 114827.5231 | test MSE 237450.8121
epoch 160 | train MSE 17120.6920 | val MSE 167617.2439 | test MSE 360991.2404
epoch 170 | train MSE 38591.8357 | val MSE 155925.9459 | test MSE 261809.3922
epoch 180 | train MSE 57447.8566 | val MSE 183334.7059 | test MSE 278808.1949
epoch 190 | train MSE 18137.6860 | val MSE 123128.2081 | test MSE 248502.9620
epoch 200 | train MSE 17628.0502 | val MSE 140998.0592 | test MSE 283774.0205
epoch 210 | train MSE 28319.9277 | val MSE 121294.0707 | test MSE 218610.4429
epoch 220 | train MSE 28242.0575 | val MSE 121446.0613 | test MSE 228927.6306
epoch 230 | train MSE 38406.1204 | val MSE 178649.5398 | test MSE 277631.2728
early stop at epoch 230

=== Per-distance MSE (raw units, best-val checkpoint) ===
  band              n         MSE    const baseline
  dist == 0         1   9740.1598         9175.8486
  dist == 1      1454  22728.0372      3155420.7500
  dist =

# HGNN

In [53]:
def add_virtual_master(edge_index: torch.Tensor, num_real_nodes: int):
    """
    Append a virtual master node at index num_real_nodes. Add bidirectional
    edges between the master and every real node.

    Returns: new edge_index (with master edges added), new num_nodes.
    """
    master_idx = num_real_nodes
    real_ids = torch.arange(num_real_nodes, device=edge_index.device)
    master_col = torch.full((num_real_nodes,), master_idx, device=edge_index.device, dtype=torch.long)

    # real → master and master → real
    real_to_master = torch.stack([real_ids, master_col], dim=0)
    master_to_real = torch.stack([master_col, real_ids], dim=0)

    new_edge_index = torch.cat([edge_index, real_to_master, master_to_real], dim=1)
    return new_edge_index, num_real_nodes + 1


# HGNN Model

In [54]:
class HGNN_VMN(torch.nn.Module):
    """
    Same SAGE architecture as the baseline, but operates on the augmented
    graph that includes a virtual master node.

    The master gets its own learned initial embedding (it has no real
    features), while real nodes use their LIWC features as before.
    """

    def __init__(self, in_dim: int, hidden_dim: int = 128, num_layers: int = 3, dropout: float = 0.3):
        super().__init__()
        self.dropout = dropout

        # Project real-node features into the same space as the master embedding
        self.input_proj = torch.nn.Linear(in_dim, hidden_dim)
        # Learned initial embedding for the master node
        self.master_init = torch.nn.Parameter(torch.zeros(1, hidden_dim))
        torch.nn.init.normal_(self.master_init, std=0.1)

        self.convs = torch.nn.ModuleList([
            SAGEConv(hidden_dim, hidden_dim) for _ in range(num_layers)
        ])
        self.head = torch.nn.Linear(hidden_dim, 1)

    def forward(self, x_real, edge_index_aug):
        # Project real features and append master init
        h_real = self.input_proj(x_real)            # [N, H]
        h = torch.cat([h_real, self.master_init], dim=0)  # [N+1, H]

        for i, conv in enumerate(self.convs):
            h = conv(h, edge_index_aug)
            if i < len(self.convs) - 1:
                h = F.relu(h)
                h = F.dropout(h, p=self.dropout, training=self.training)

        # Drop the master from the output — we only predict on real nodes
        return self.head(h[:-1]).squeeze(-1)



# Training

In [55]:
def train_hgnn_vmn(data, train_mask, val_mask, test_mask, dist, device=DEVICE):
    """Same protocol as train_baseline: same standardization, same optimizer
    settings, same early stopping. Only the model and edge_index differ."""
    data = data.to(device)
    train_mask = train_mask.to(device)
    val_mask = val_mask.to(device)
    test_mask = test_mask.to(device)
    dist = dist.to(device)

    # Symmetrize real-graph edges, then add the virtual master.
    real_edges = to_undirected(data.edge_index, num_nodes=data.num_nodes)
    edge_index_aug, _ = add_virtual_master(real_edges, data.num_nodes)
    edge_index_aug = edge_index_aug.to(device)

    # Constant baseline (raw units) for reference
    y_raw = data.y.clone()
    const_test = constant_baseline_mse(y_raw, train_mask, test_mask)
    print(f"\nConstant-predictor test MSE (raw units): {const_test:.2f}")

    # Standardize using train-only stats (same as baseline)
    data.x, data.y, _, y_std = standardize_with_train_stats(data.x, data.y, train_mask)

    model = HGNN_VMN(
        in_dim=data.x.size(1), hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS
    ).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    masks = {"train": train_mask, "val": val_mask, "test": test_mask}

    @torch.no_grad()
    def eval_all():
        model.eval()
        pred = model(data.x, edge_index_aug)
        return {
            name: F.mse_loss(pred[mask], data.y[mask]).item() * (y_std ** 2)
            for name, mask in masks.items()
        }

    best_val = float("inf")
    best_state = None
    epochs_since_improve = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        opt.zero_grad()
        pred = model(data.x, edge_index_aug)
        loss = F.mse_loss(pred[train_mask], data.y[train_mask])
        loss.backward()
        opt.step()

        if epoch % 10 == 0 or epoch == 1:
            scores = eval_all()
            print(f"epoch {epoch:3d} | train {scores['train']:.2f} | "
                  f"val {scores['val']:.2f} | test {scores['test']:.2f}")
            if scores["val"] < best_val - 1e-5:
                best_val = scores["val"]
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                epochs_since_improve = 0
            else:
                epochs_since_improve += 10
                if epochs_since_improve >= PATIENCE:
                    print(f"early stop at epoch {epoch}")
                    break

    if best_state is not None:
        model.load_state_dict(best_state)

    # Per-band MSE — adapted from per_band_mse but with augmented edges
    print("\n=== HGNN-VMN per-distance MSE (raw units, best-val) ===")
    model.eval()
    with torch.no_grad():
        pred = model(data.x, edge_index_aug)
    rows = []
    for d in [0, 1, 2, 3, 4]:
        band = (dist == d) & data.is_source
        if band.sum() == 0:
            continue
        mse = F.mse_loss(pred[band], data.y[band]).item() * (y_std ** 2)
        rows.append((f"dist == {d}", int(band.sum()), mse))
    band = (dist >= 5) & data.is_source
    if band.sum() > 0:
        mse = F.mse_loss(pred[band], data.y[band]).item() * (y_std ** 2)
        rows.append(("dist >= 5", int(band.sum()), mse))

    for band_name, n, mse in rows:
        print(f"  {band_name:<11s}  n={n:>6d}  MSE={mse:.2f}")

    return model, y_std, rows


# Running & Comparison

In [56]:
import json
import collections

SEEDS_HGNN = list(range(N_SEEDS))


data_original = load_graph_with_source_mask(GRAPH_PATH, target_idx=TARGET_IDX)

all_runs_hgnn = []
for seed in SEEDS_HGNN:
    if seed % 10 == 0:
        print(f"\n--- HGNN seed {seed} ---")
    torch.manual_seed(seed)

    data_run = Data(
        x=data_original.x.clone(),
        y=data_original.y.clone(),
        edge_index=data_original.edge_index.clone(),
    )
    data_run.num_nodes = data_original.num_nodes
    data_run.is_source = data_original.is_source.clone()

    # Suppress the per-epoch printing for seeds > 0 — too noisy for 100 runs
    import io, contextlib
    if seed == 0:
        model, y_std, rows = train_hgnn_vmn(data_run, train_mask, val_mask, test_mask, dist)
    else:
        with contextlib.redirect_stdout(io.StringIO()):
            model, y_std, rows = train_hgnn_vmn(data_run, train_mask, val_mask, test_mask, dist)

    all_runs_hgnn.append({"seed": seed, "rows": rows, "y_std": y_std})

# Aggregate
print(f"\n{'='*70}")
print(f"  HGNN-VMN AGGREGATE ACROSS {len(SEEDS_HGNN)} SEEDS")
print(f"{'='*70}\n")

band_mses_hgnn = collections.defaultdict(list)
band_ns_hgnn = {}
for run in all_runs_hgnn:
    for band_name, n, mse in run["rows"]:
        band_mses_hgnn[band_name].append(mse)
        band_ns_hgnn[band_name] = n

# Load baseline for side-by-side comparison
with open("/content/drive/MyDrive/Thesis /baseline_results.json") as f:
    baseline = json.load(f)
baseline_by_band = {row["band"]: row for row in baseline["per_band"]}

print(f"  {'band':<11s}  {'n':>6s}  {'baseline mean ± std':>22s}  "
      f"{'HGNN mean ± std':>22s}  {'Δ mean':>10s}  {'Δ /σ_base':>10s}")
print(f"  {'-'*11}  {'-'*6}  {'-'*22}  {'-'*22}  {'-'*10}  {'-'*10}")

summary_rows_hgnn = []
for band_name in ["dist == 0", "dist == 1", "dist == 2", "dist == 3", "dist == 4", "dist >= 5"]:
    if band_name not in band_mses_hgnn or band_name not in baseline_by_band:
        continue
    mses = band_mses_hgnn[band_name]
    n = band_ns_hgnn[band_name]
    h_mean = sum(mses) / len(mses)
    h_std = (sum((m - h_mean) ** 2 for m in mses) / len(mses)) ** 0.5

    b = baseline_by_band[band_name]
    b_mean, b_std = b["mean_mse"], b["std_mse"]

    delta = b_mean - h_mean
    sigma_units = delta / b_std if b_std > 0 else float("nan")

    print(f"  {band_name:<11s}  {n:>6d}  "
          f"{b_mean:>10.0f} ± {b_std:>7.0f}  "
          f"{h_mean:>10.0f} ± {h_std:>7.0f}  "
          f"{delta:>+10.0f}  {sigma_units:>+10.2f}")

    summary_rows_hgnn.append({
        "band": band_name, "n": n,
        "mean_mse": h_mean, "std_mse": h_std,
        "min_mse": min(mses), "max_mse": max(mses),
        "delta_vs_baseline": delta,
        "sigma_units": sigma_units,
    })

# Save
results_hgnn = {
    "model": "HGNN_VMN_3layer",
    "target_idx": TARGET_IDX,
    "anchor": int(anchor),
    "seeds": SEEDS_HGNN,
    "per_band": summary_rows_hgnn,
}
out_path = "/content/drive/MyDrive/Thesis /hgnn_vmn_results.json"
with open(out_path, "w") as f:
    json.dump(results_hgnn, f, indent=2)
print(f"\nSaved to {out_path}")



--- HGNN seed 0 ---

Constant-predictor test MSE (raw units): 9481496.00
epoch   1 | train 2962191.24 | val 6586860.81 | test 9045800.43
epoch  10 | train 361339.89 | val 673877.56 | test 816626.95
epoch  20 | train 178019.80 | val 555418.04 | test 916408.95
epoch  30 | train 97902.00 | val 228330.32 | test 341828.41
epoch  40 | train 122113.03 | val 402134.87 | test 702703.82
epoch  50 | train 196711.07 | val 520121.19 | test 711140.30
epoch  60 | train 62954.27 | val 163041.48 | test 230416.34
epoch  70 | train 33927.63 | val 122086.42 | test 206762.63
epoch  80 | train 38486.99 | val 140017.82 | test 307977.60
epoch  90 | train 59053.27 | val 161899.52 | test 225153.89
epoch 100 | train 20524.07 | val 84441.56 | test 164924.60
epoch 110 | train 56710.07 | val 139782.70 | test 188224.58
epoch 120 | train 48068.54 | val 192042.71 | test 282094.18
epoch 130 | train 62225.10 | val 181732.49 | test 236267.97
epoch 140 | train 50068.45 | val 166368.96 | test 226580.84
epoch 150 | train 1